In [1]:
import cv2
import os

# --- 1. CONFIGURATION ---
IMG_PATH = "Annotated_Better_Again/101142060_00003.jpg"
LABEL_PATH = "Annotated_Better_Again/labels/101142060_00003.txt"

# --- 2. GLOBAL RULES DICTIONARY ---
# Key = The LINE NUMBER in your text file (0 to 35)
# Value = The math to transform that specific anchor into its handwriting box.
#
# TERMS:
# x_fac / y_fac:  Shift relative to anchor size (1.0 = shift one full width/height)
# w_fac / h_fac:  Resize relative to anchor size (2.0 = double the size)
# x_pix / y_pix:  Fine-tuning shift in raw pixels (useful for fixed alignments)

#Psuedocode the rules
#rule 1: (from x-max of label 1 to x-min of label 2 -3*w of label 2, y-min of label 1 to y-max of label 1 *2.15)
#rule 2: (x-max of label 2 to end of document, y-max of label 2 *2.5)
#rule 3: (x-max of label 3 to label 4 x-min, no change in y)
#rule 4: (x-max to end of document, no change in y)
#rule 5,6,7,8,15,16,17,18: (no change in x, y-min and y-max - height or vertical shift downwards of the same size as the label)
#rule 9: (x-max of label 9 to label 10 x-min y-min of label 9 to y-min of label 5 - height of label 5)
#rule 10: (x-max of label 10 to x-min of label 19, y-min of label 10 to y-min of label 8 - height of label 8)
#rule 11: (x-max of label 11 to x-min of label 21, y-min of label 11 to y-min of label 10)
#rule 12: (x-max of label 12 to x-min of label 22, y-min of label 12 to y-min of label 11)
#rule 13: (x-max of label 13 to x-min of label 23, y-min of label 13 to y-min of label 12)
#rule 14: (x-max of label 14 to x-min of label 24, y-min of label 14 to y-min of label 13)
#rule 19: (x-max of label 19 to x-min of label 20, y-min of label 19 to y-min of label 15 - height of label 8)
#rule 20: (x-max of label 20 to end of document, y-min of label 20 to y-min of label 18 - height of label 8)
#rule 21: (x-max of label 21 to end of document, y-min of label 21 to y-min of label 20)
#rule 22: (x-max of label 22 to end of document, y-min of label 22 to y-min of label 21)
#rule 23: (x-max of label 23 to end of document, y-min of label 23 to y-min of label 22)
#rule 24: (x-max of label 24 to end of document, y-min of label 24 to y-min of label 23)
#rule 25: (x-max of label 25 to x-min of label 26, y-min of label 25 to y-min of label 14)
#rule 26: (x-max of label 26 to end of document, y-min of label 26 to y-min of label 24)
#rule 27: (x-min of label 27 to -end of document y-max*2)
#rule 28: (x-max of label 27 to x-min label 28, y-max*2)
#rule 29: (x-max of label 29 to x-min of label 30, y-max*1.25)
#rule 30: (x-max of label 30 to x-min of label 31, y-max*1.25)
#rule 31: (x-max of label 32 to x-min of label 33, y-max*1.2)
#rule 32: (x-max of label 34 to x-min of label 35, y-max*1.2)
#rule 33: (x-max of label 35 to x-min of label 36, y-max*1.15)
#rule 34: (x-max of label 37 to end of document, y-max*1.2)



RULES = {
    # --- Index 0 (First box in file) ---
    # Example: "Date" label. Handwriting is to the RIGHT.
    0: {"x_fac": 1.2, "y_fac": 0.0, "w_fac": 3.0, "h_fac": 1.5, "x_pix": 0, "y_pix": 0},

    # --- Index 1 (Second box in file) ---
    # Example: "Invoice #" label. Handwriting is BELOW.
    1: {"x_fac": 0.0, "y_fac": 1.5, "w_fac": 1.0, "h_fac": 2.0, "x_pix": 10, "y_pix": 0},

    # --- Index 2 ---
    # Example: "Total". Handwriting is HUGE and centered.
    2: {"x_fac": 0.0, "y_fac": 0.0, "w_fac": 4.0, "h_fac": 4.0, "x_pix": 0, "y_pix": 0},

    # ... You can copy-paste this line 36 times and tweak each one ...
    3: {"x_fac": 1.0, "y_fac": 0.0, "w_fac": 1.0, "h_fac": 1.0, "x_pix": 0, "y_pix": 0},
}

# Fallback rule for indices you haven't defined yet
DEFAULT_RULE = {"x_fac": 1.0, "y_fac": 0.0, "w_fac": 1.0, "h_fac": 1.0, "x_pix": 0, "y_pix": 0}


def apply_math_overlay(image, label_lines):
    h_img, w_img = image.shape[:2]
    
    # Loop through lines by INDEX (i)
    for i, line in enumerate(label_lines):
        parts = line.strip().split()
        
        # Parse YOLO format (class x_c y_c w h)
        # We ignore parts[0] (class) because you said they are all the same
        norm_xc, norm_yc, norm_w, norm_h = map(float, parts[1:])
        
        # 1. Get Anchor Coordinates (Pixels)
        anchor_w = norm_w * w_img
        anchor_h = norm_h * h_img
        anchor_xc = norm_xc * w_img
        anchor_yc = norm_yc * h_img
        
        # 2. Get the Rule for this specific index
        rule = RULES.get(i, DEFAULT_RULE)
        
        # 3. CALCULATE TARGET (THE EQUATION)
        # New Center = Old Center + (Old Size * Factor) + Pixel Offset
        target_xc = anchor_xc + (anchor_w * rule["x_fac"]) + rule["x_pix"]
        target_yc = anchor_yc + (anchor_h * rule["y_fac"]) + rule["y_pix"]
        
        # New Size = Old Size * Factor
        target_w = anchor_w * rule["w_fac"]
        target_h = anchor_h * rule["h_fac"]
        
        # 4. Convert to Corners for Drawing
        # Anchor (Green)
        ax1 = int(anchor_xc - anchor_w/2)
        ay1 = int(anchor_yc - anchor_h/2)
        ax2 = int(anchor_xc + anchor_w/2)
        ay2 = int(anchor_yc + anchor_h/2)
        
        # Target (Red)
        tx1 = int(target_xc - target_w/2)
        ty1 = int(target_yc - target_h/2)
        tx2 = int(target_xc + target_w/2)
        ty2 = int(target_yc + target_h/2)
        
        # 5. Draw
        cv2.rectangle(image, (ax1, ay1), (ax2, ay2), (0, 255, 0), 2) # Green Anchor
        cv2.rectangle(image, (tx1, ty1), (tx2, ty2), (0, 0, 255), 2) # Red Target
        
        # Draw ID Label so you know which Rule to edit
        label_text = f"ID: {i}"
        cv2.putText(image, label_text, (ax1, ay1 - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
        # Draw Line connecting them
        cv2.line(image, (int(anchor_xc), int(anchor_yc)), 
                 (int(target_xc), int(target_yc)), (255, 255, 0), 1)

    return image

In [2]:
def main():
    if not os.path.exists(IMG_PATH) or not os.path.exists(LABEL_PATH):
        print("Error: Files not found.")
        return

    # Load Data
    original_img = cv2.imread(IMG_PATH)
    with open(LABEL_PATH, "r") as f:
        lines = f.readlines()
        
    print(f"Loaded {len(lines)} labels. Processing by index order...")
    
    # Process
    overlay_img = apply_math_overlay(original_img.copy(), lines)
    
    # Show
    cv2.namedWindow("Equation Tuner", cv2.WINDOW_NORMAL)
    cv2.imshow("Equation Tuner", overlay_img)
    
    print("Showing overlay. Press any key to exit.")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Loaded 37 labels. Processing by index order...
Showing overlay. Press any key to exit.


In [ ]:
import cv2

# --- 1. THE DATA STRUCTURE ---
class Box:
    """A simple container to make your math readable."""
    def __init__(self, x1, y1, x2, y2):
        self.x1 = int(x1)
        self.y1 = int(y1)
        self.x2 = int(x2)
        self.y2 = int(y2)
        self.w = self.x2 - self.x1
        self.h = self.y2 - self.y1
        self.xc = self.x1 + (self.w // 2)
        self.yc = self.y1 + (self.h // 2)

# --- 2. YOUR 34 MATH RULES ---
# Input parameters for every rule: (anchors_dict, image_width, image_height)
# Returns: (x1, y1, x2, y2) of the handwriting ROI

EXTRACTION_RULES = {
    # Rule 1: Anchor the BOTTOM edge (y2), stretch UPWARD by 2.15x height
    "Field_1": lambda a, w, h: (a[0].x2, a[0].y2 - (a[0].h * 2), a[1].x1 - (3 * a[1].w), a[0].y2),

    # Rule 2: Anchor the BOTTOM edge (y2), stretch UPWARD by 2.5x height
    "Field_2": lambda a, w, h: (a[1].x2, a[1].y2 - (a[1].h * 2.25), w, a[1].y2),

    # Rule 3: x-max of 2 to x-min of 3. No change in y.
    "Field_3": lambda a, w, h: (a[2].x2, a[2].y1, a[3].x1, a[2].y2),

    # Rule 4: x-max of 3 to end of document. No change in y.
    "Field_4": lambda a, w, h: (a[3].x2, a[3].y1, w, a[3].y2),

    # Rules 5, 6, 7, 8: No change in x. Vertical shift downwards of the same size as the label.
    "Field_5": lambda a, w, h: (a[4].x1, a[4].y2, a[4].x2, a[4].y2 + a[4].h),
    "Field_6": lambda a, w, h: (a[5].x1, a[5].y2, a[5].x2, a[5].y2 + a[5].h),
    "Field_7": lambda a, w, h: (a[6].x1, a[6].y2, a[6].x2, a[6].y2 + a[6].h),
    "Field_8": lambda a, w, h: (a[7].x1, a[7].y2, a[7].x2, a[7].y2 + a[7].h),

    # Rule 9: Anchor bottom to label 9 (index 8). Top stretches up to label 5's top.
    "Field_9": lambda a, w, h: (a[8].x2, a[4].y2 + a[4].h, a[9].x1, a[8].y2),

    # Rule 10: Anchor bottom to label 10 (index 9). Top stretches up to label 8's top.
    "Field_10": lambda a, w, h: (a[9].x2, a[7].y2 + a[7].h, a[18].x1, a[9].y2),

    # Rule 11: Bottom is label 11's bottom. Top is label 10's bottom.
    "Field_11": lambda a, w, h: (a[10].x2, a[9].y2, a[20].x1, a[10].y2),

    # Rule 12: Bottom is label 12's bottom. Top is label 11's bottom.
    "Field_12": lambda a, w, h: (a[11].x2, a[10].y2, a[21].x1, a[11].y2),

    # Rule 13: Bottom is label 13's bottom. Top is label 12's bottom.
    "Field_13": lambda a, w, h: (a[12].x2, a[11].y2, a[22].x1, a[12].y2),

    # Rule 14: Bottom is label 14's bottom. Top is label 13's bottom.
    "Field_14": lambda a, w, h: (a[13].x2, a[12].y2, a[23].x1, a[13].y2),

    # Rules 15, 16, 17, 18: Shift down directly underneath the anchor.
    "Field_15": lambda a, w, h: (a[14].x1, a[14].y2, a[14].x2, a[14].y2 + a[14].h),
    "Field_16": lambda a, w, h: (a[15].x1, a[15].y2, a[15].x2, a[15].y2 + a[15].h),
    "Field_17": lambda a, w, h: (a[16].x1, a[16].y2, a[16].x2, a[16].y2 + a[16].h),
    "Field_18": lambda a, w, h: (a[17].x1, a[17].y2, a[17].x2, a[17].y2 + a[17].h),

   # Rule 19: Bottom is label 19's bottom. Top stretches up.
    "Field_19": lambda a, w, h: (a[18].x2, a[15].y2 + a[15].h, a[19].x1, a[18].y2),

    # Rule 20: Bottom is label 20's bottom. Top stretches up.
    "Field_20": lambda a, w, h: (a[19].x2, a[17].y2 + a[17].h, w, a[19].y2),

    # Rule 21: Bottom is label 21's bottom. Top is label 20's bottom.
    "Field_21": lambda a, w, h: (a[20].x2, a[19].y2, w, a[20].y2),

    # Rule 22: Bottom is label 22's bottom. Top is label 21's bottom.
    "Field_22": lambda a, w, h: (a[21].x2, a[20].y2, w, a[21].y2),

    # Rule 23: x-max of 22 to end of document. y-min of 22 to y-min of 21.
    "Field_23": lambda a, w, h: (a[22].x2, a[22].y2, w, a[21].y2),

    # Rule 24: x-max of 23 to end of document. y-min of 23 to y-min of 22.
    "Field_24": lambda a, w, h: (a[23].x2, a[23].y2, w, a[22].y2),

    # Rule 25: x-max of 24 to x-min of 25. y-min of 24 to y-min of 14.
    "Field_25": lambda a, w, h: (a[24].x2, a[24].y1, a[25].x1, a[14].y1),

    # Rule 26: x-max of 25 to end of document. y-min of 25 to y-min of 23.
    "Field_26": lambda a, w, h: (a[25].x2, a[25].y2, w, a[23].y2),

    # Rule 27: Left edge of document (0) to x-min of 26. Anchor bottom, stretch UP 2x height.
    "Field_27": lambda a, w, h: (0+(1/9 * w), a[26].y2 - (a[26].h * 1.5), a[26].x1, a[26].y2),

    # Rule 28: Anchor bottom, stretch UP 2x height.
    "Field_28": lambda a, w, h: (a[26].x2, a[26].y2 - (a[26].h * 1.5), a[27].x1, a[26].y2),

    # Rule 29: Anchor bottom, stretch UP 1.25x height.
    "Field_29": lambda a, w, h: (a[28].x2, a[28].y2 - (a[28].h * 1.25), a[29].x1, a[28].y2),

    # Rule 30: Anchor bottom, stretch UP 1.25x height.
    "Field_30": lambda a, w, h: (a[29].x2, a[29].y2 - (a[29].h * 1.25), a[30].x1, a[29].y2),

    # Rule 31: Anchor bottom, stretch UP 1.2x height.
    "Field_31": lambda a, w, h: (a[31].x2, a[31].y2 - (a[31].h * 1.2), a[32].x1, a[31].y2),

    # Rule 32: Anchor bottom, stretch UP 1.2x height.
    "Field_32": lambda a, w, h: (a[33].x2, a[33].y2 - (a[33].h * 1.22), a[34].x1, a[33].y2),

    # Rule 33: Anchor bottom, stretch UP 1.15x height.
    "Field_33": lambda a, w, h: (a[34].x2, a[34].y2 - (a[34].h * 1.15), a[35].x1, a[34].y2),

    # Rule 34: Anchor bottom, stretch UP 1.2x height.
    "Field_34": lambda a, w, h: (a[36].x2, a[36].y2 - (a[36].h * 1.77), w, a[36].y2)
}

def load_anchors(label_lines, img_w, img_h):
    """Converts YOLO text lines into your Box dictionary."""
    anchors = {}
    for index, line in enumerate(label_lines):
        parts = line.strip().split()
        norm_xc, norm_yc, norm_w, norm_h = map(float, parts[1:])
        
        # Convert YOLO to Absolute Corners
        w_pix = norm_w * img_w
        h_pix = norm_h * img_h
        xc_pix = norm_xc * img_w
        yc_pix = norm_yc * img_h
        
        x1 = xc_pix - (w_pix / 2)
        y1 = yc_pix - (h_pix / 2)
        x2 = xc_pix + (w_pix / 2)
        y2 = yc_pix + (h_pix / 2)
        
        # Store in dictionary using the line index (0 to 36)
        anchors[index] = Box(x1, y1, x2, y2)
        
    return anchors

def test_overlay(image_path, label_path):
    image = cv2.imread(image_path)
    img_h, img_w = image.shape[:2]
    
    with open(label_path, "r") as f:
        lines = f.readlines()
        
    # 1. Load all 37 anchors into memory
    anchors = load_anchors(lines, img_w, img_h)
    
    # 2. Draw the Anchors (Green) so you can see their IDs
    for idx, box in anchors.items():
        cv2.rectangle(image, (box.x1, box.y1), (box.x2, box.y2), (0, 255, 0), 1)
        cv2.putText(image, str(idx), (box.x1, box.y1 - 2), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 1)

    # 3. Apply your 34 Rules to generate the Targets (Red)
    for field_name, rule_math in EXTRACTION_RULES.items():
        try:
            # Execute the lambda function
            tx1, ty1, tx2, ty2 = rule_math(anchors, img_w, img_h)
            
            # Draw Target ROI
            cv2.rectangle(image, (int(tx1), int(ty1)), (int(tx2), int(ty2)), (0, 0, 255), 2)
            cv2.putText(image, field_name, (int(tx1), int(ty1) + 15), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            
        except KeyError as e:
            print(f"Error calculating {field_name}: Anchor {e} not found.")

    cv2.namedWindow("Extraction Lab", cv2.WINDOW_NORMAL)
    # 2. Force the window to a specific size. 
    # If 1280x1280 is still too tall for your screen, change this to 800, 800
    cv2.resizeWindow("Extraction Lab", 1000, 1000)
    cv2.imshow("Extraction Lab", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

test_overlay("Annotated_Better_Again/101142060_00004.jpg", "Annotated_Better_Again/labels/101142060_00004.txt")